# Ordered Logistic Regression Results Dataset Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://mlcroissant.io/) library. The dataset includes ordered logistic regression outputs for adoption predictors of indigenous and modern knowledge in rangeland management among pastoral households in Northern Kenya.

### Dataset Source
The dataset is described with a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Install mlcroissant, if not already available
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and initialize the Croissant dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Explore the metadata (accessed as attributes, not subscripted)
meta = dataset.metadata
print(f"Dataset name: {meta.name}\n\nDescription: {meta.description}\n")
print(f"Published: {meta.datePublished}")
print(f"Identifier: {meta.identifier}")

## 2. Data Overview
Review available record sets, their IDs, and fields. All are referenced by their `@id` according to the Croissant specification.

Let's enumerate all available record sets, the associated field `@id`s, and the distribution objects (files).

In [ ]:
# List all available record sets and their fields by @id.
record_set_ids = []

# dataset.metadata.recordSet may be missing or empty, so we must check.
if hasattr(meta, 'recordSet') and meta.recordSet:
    for rs in meta.recordSet:
        print(f"Record Set: {rs['@id']}")
        record_set_ids.append(rs['@id'])
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                print(f"    {field['@id']} (type: {field.get('dataType', 'n/a')})")
        print()
else:
    # If recordSet not populated in JSON-LD, try listing available record sets through the Dataset API (if supported)
    print("No record sets found in metadata. Attempting to enumerate programmatically.")
    # mlcroissant 0.7.0+ exposes dataset.record_sets
    if hasattr(dataset, 'record_sets'):
        for rs_id, rs in dataset.record_sets.items():
            print(f"Record Set: {rs_id}")
            record_set_ids.append(rs_id)
            if hasattr(rs, 'fields'):
                print("  Fields:")
                for field in rs.fields:
                    print(f"    {field['@id']} (type: {field.get('dataType', 'n/a')})")
            print()
    else:
        print("Warning: No record sets available in metadata or via API.")

# For this dataset, if no recordSet in metadata, we will introspect using dataset.record_sets or dataset.records API
# For demonstration, let's attempt to infer the available record sets from the Croissant schema JSON, if needed.

In [ ]:
# If record_set_ids is empty after previous step, try discovering record set IDs using dataset interface
if not record_set_ids:
    # In mlcroissant 0.7.0+, use dataset.record_sets.keys(), otherwise try a default
    if hasattr(dataset, 'record_sets'):
        record_set_ids = list(dataset.record_sets.keys())
        print(f"Discovered record sets: {record_set_ids}")
    else:
        # If mlcroissant internals not exposed, try to load at least one (commonly the first)
        record_set_ids = ["cr:RecordSet"]
        print(f"Using generic record set @id: {record_set_ids[0]}")

# For this dataset, let's display a preview of individual records for the first record set:
sample_record_set_id = record_set_ids[0]
print(f"Displaying a few example records for record set: {sample_record_set_id}\n")
try:
    for i, record in enumerate(dataset.records(record_set=sample_record_set_id)):
        print(json.dumps(record, indent=2))
        if i >= 2:
            break
except Exception as e:
    print(f"Could not load records for {sample_record_set_id}: {e}")


## 3. Data Extraction
Load the records from the available record sets into Pandas DataFrames for further analysis. Ensure you use the `@id` for each record set and (if desired) for the field columns as keys.

In [ ]:
# Extract all discovered record sets into DataFrames indexed by their @id
dataframes = {}
for r_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=r_id))
        df = pd.DataFrame(records)
        dataframes[r_id] = df
        print(f"Loaded record set {r_id} with {len(df)} records and columns: \n{list(df.columns)}\n")
    except Exception as err:
        print(f"Error loading record set {r_id}: {err}")

# Preview the first few rows of the first available record set
sample_df = dataframes[record_set_ids[0]]
sample_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's work with the main record set. First, we'll identify numeric and categorical fields using their `@id` (column name). We'll demonstrate filtering, normalization, and grouping operations.

**All columns are referenced strictly by their `@id`.**

In [ ]:
# For demonstration, select a numeric column for analysis.
# We'll automatically pick the first numeric-looking column if possible.
df = dataframes[record_set_ids[0]]

# Try to infer a numeric field by checking dtypes or known names
numeric_field_id = None
group_field_id = None

# Look for columns like 'log_likelihood', 'coefficient', or similar
for col in df.columns:
    if df[col].dtype in ('float64', 'int64', 'float32', 'int32'):
        numeric_field_id = col
        break
    elif ('log' in col.lower() and 'likelihood' in col.lower()) or 'coefficient' in col.lower():
        # fallback: check column name string
        numeric_field_id = col
        break

# Try to pick a grouping field
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() < len(df)//4 and df[col].dtype == object:
        group_field_id = col
        break

if numeric_field_id is not None:
    print(f"Using '{numeric_field_id}' as numeric field for filtering and normalization.")
else:
    print("No numeric field available in the selected record set.")

# Filtering: values greater than a threshold (use mean if possible)
if numeric_field_id is not None and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.3f} (filtered {len(filtered_df)} records).")
    print(filtered_df.head())

    # Normalization
    norm_field = f"{numeric_field_id}_normalized"
    filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized column '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, norm_field]].head())

    # Grouping (if there is a suitable group-by column)
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped by '{group_field_id}' (showing mean {numeric_field_id} per group):")
        print(grouped_df.head())
    else:
        print("\nNo suitable group field was found.")
else:
    print("No numeric field suitable for EDA found in the loaded data.")

## 5. Visualization
Plot the distribution of the selected numeric field, and if a group field exists, visualize differences by group.

All visualizations refer to columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, color='dodgerblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id is not None:
        # Boxplot/grouped distribution
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, showfliers=False)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, explore, and process the "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management" dataset using the `mlcroissant` library.

- All data entities were referenced strictly by `@id` as defined in the Croissant schema.
- The notebook showcased how to identify numeric and grouping variables, filter and normalize data, and visualize field distributions directly from the Croissant-described data package.

You can further extend this exploration depending on your analytical or scientific objectives.